In [40]:
import platform
import psutil
import os
import time

from pathlib import Path

import tensorflow as tf
import keras

import numpy as np
import pandas as pd

BASE_DIR = Path().cwd().parent

PICTURE_DATA_DIR = BASE_DIR / "picture_data"
DATASET_DIR = PICTURE_DATA_DIR / "start_bilder"

SELECTED_CLASSES = ["brille", "schuh"]

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42
EPOCHS = 3

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Verfügbare GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
Keras: 3.15.1
Verfügbare GPUs: []


In [41]:
process = psutil.Process(os.getpid())

def print_memory_usage():

    system_ram = psutil.virtual_memory()
    process_ram = process.memory_info()

    print(f"System RAM: {system_ram.percent:.1f} %")
    print(f"System verfügbar: {system_ram.available / 1024**3:.2f} GB")
    print(f"Python Prozess: {process_ram.rss / 1024**3:.2f} GB")

In [42]:
print("CPU:", platform.processor())
print("Physische Kerne:", psutil.cpu_count(logical=False))
print("Logische Kerne:", psutil.cpu_count(logical=True))
print("RAM:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")

CPU: AMD64 Family 23 Model 160 Stepping 0, AuthenticAMD
Physische Kerne: 4
Logische Kerne: 8
RAM: 13.83 GB


In [43]:
train_ds, test_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    class_names=SELECTED_CLASSES,
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="both",
    seed=SEED,
    shuffle=True
)

print("Klassen:", train_ds.class_names)

Found 141 files belonging to 2 classes.
Using 113 files for training.
Using 28 files for validation.
Klassen: ['brille', 'schuh']


- image_dataset_from_directory erzeugt tensorflow-datasets
- diese sind über tf.data adressierbar
- über class_names kann gesteuert werden, welche Unterordner von DATASET_DIR als Klassen verwendet werden

In [44]:
preprocess_input = keras.applications.resnet50.preprocess_input

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(tf.data.AUTOTUNE)

test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).cache().prefetch(tf.data.AUTOTUNE)

Preprocessing bearbeitet Bilder folgendermaßen:
- Umkehrung der Reihenfolge der Farbkanäle von RGB auf BGR
- Zero-Centering: von jedem Farbkanal wird ein festgelegter Mittelwert abgezogen
- Keine zusätzliche Skalierung auf [0,1] nötig

In [45]:
start = time.perf_counter()

train_samples = sum(int(x.shape[0]) for x, y in train_ds)

test_samples = sum(int(x.shape[0]) for x, y in test_ds)

data_loading_time = time.perf_counter() - start

print(f"Trainingsbilder: {train_samples}")
print(f"Testbilder: {test_samples}")
print(f"Datenvorbereitung: {data_loading_time:.2f} Sekunden")

Trainingsbilder: 113
Testbilder: 28
Datenvorbereitung: 0.30 Sekunden


- jedes Element aus train_ds besteht aus
    - mehrdimensionalem Bildvektor (batch_size, spatial_x, spatial_y, Dimension der Bildvektoren (z.B. 3 Farbkanäle))
    - Class Label

In [46]:
for x, y in train_ds:
    print(x.shape)

(16, 224, 224, 3)
(16, 224, 224, 3)
(16, 224, 224, 3)
(16, 224, 224, 3)
(16, 224, 224, 3)
(16, 224, 224, 3)
(16, 224, 224, 3)
(1, 224, 224, 3)


In [47]:
start = time.perf_counter()

with tf.device("/CPU:0"):

    base_model = keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))

    x = base_model(inputs, training=False)

    x = keras.layers.GlobalAveragePooling2D()(x)

    outputs = keras.layers.Dense(2, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

model_creation_time = time.perf_counter() - start

model.summary()

print(f"Modellaufbau: {model_creation_time:.2f} Sekunden")
print(
    "Trainierbare Parameter:", 
    sum(np.prod(v.shape) for v in model.trainable_variables)
)
print("Gesamte Parameter:", model.count_params())

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │         4,098 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,591,810 (90.00 MB)

 Trainable params: 4,098 (16.01 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Modellaufbau: 2.58 Sekunden
Trainierbare Parameter: 4098
Gesamte Parameter: 23591810


- Imagenet: Datensatz auf dem die verwendeten Gewichte trainiert wurden
- include_top=False: entfernt den Klassifikationslayer
- base_model.trainable = False: friert die Basis ein
- Input() erzeugt einen symbolischen Keras Eingabetensor, der die erwartete Datenstruktur vorgibt
- GlobalAveragePooling2D() wird an das BaseModel verkettet
    - erzeugt aus 7x7 Werten der 2048 Feature Maps einen einzelnen Mittelwert
- Dense() wird an das BaseModel verkettet
    - die Anzahl der Ausgänge wird festgelegt
    - Softmax normalisiert auf Gesamtsumme 100% der Wahrscheinlichkeiten
- keras.Model(inputs, outputs) konstruiert das finale Modell
- model.compile() konfiguriert das Training
    - Adam optimiert auf Basis der berechneten Gradienten die Gewichte
    - sparse_categorical_crossentropy (Loss durch numerischen Vergleich Vorhersage vs True)

In [48]:
class EpochTimer(keras.callbacks.Callback):

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        duration = time.perf_counter() - self.epoch_start

        print(f"\nEpoche {epoch + 1}: ", f"{duration:.2f} Sekunden")
        print_memory_usage()
        print("=" * 60)
        print()
        

start = time.perf_counter()

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[EpochTimer()],
    verbose=1
)

training_time = time.perf_counter() - start

print(f"\nGesamte Trainingszeit: {training_time:.2f} Sekunden")
print(f"Durchschnitt pro Epoche: {training_time / EPOCHS:.2f} Sekunden")

Epoch 1/3


c:\Users\MarcelKramer\Desktop\Unterrichtsmaterialien\08. Machine Learning\Abschlussprojekt\cnn_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 896ms/step - accuracy: 0.7168 - loss: 0.6521
Epoche 1:  15.41 Sekunden
System RAM: 73.3 %
System verfügbar: 3.69 GB
Python Prozess: 1.31 GB

8/8 ━━━━━━━━━━━━━━━━━━━━ 15s 896ms/step - accuracy: 0.7168 - loss: 0.6521
Epoch 2/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 897ms/step - accuracy: 0.9735 - loss: 0.1131
Epoche 2:  7.39 Sekunden
System RAM: 73.3 %
System verfügbar: 3.70 GB
Python Prozess: 1.32 GB

8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 897ms/step - accuracy: 0.9735 - loss: 0.1131
Epoch 3/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 851ms/step - accuracy: 1.0000 - loss: 0.0490
Epoche 3:  6.98 Sekunden
System RAM: 73.0 %
System verfügbar: 3.74 GB
Python Prozess: 1.32 GB

8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 851ms/step - accuracy: 1.0000 - loss: 0.0490

Gesamte Trainingszeit: 29.81 Sekunden
Durchschnitt pro Epoche: 9.94 Sekunden


In [49]:
probabilities = model.predict(test_ds)

y_pred = np.argmax(probabilities, axis=1)

y_proba = [np.max(prob) for prob in probabilities]

1/2 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/stepWARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x000001ACC560F560> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step


In [54]:
y_test = np.concatenate([y.numpy() for x, y in test_ds])

pred_data = {
    "y_test": y_test,
    "y_pred": y_pred,
    "y_proba": y_proba
}

pred_df = pd.DataFrame(pred_data)

confusion_filter = pred_df["y_pred"] != pred_df["y_test"]

confusion_df = pred_df[confusion_filter]

confusion_df

,y_test,y_pred,y_proba


In [51]:
start = time.perf_counter()

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

evaluation_time = time.perf_counter() - start

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Evaluationszeit: {evaluation_time:.2f} Sekunden")

Test Loss: 0.0327
Test Accuracy: 1.0000
Evaluationszeit: 4.36 Sekunden


In [52]:
total_time = data_loading_time + model_creation_time + training_time + evaluation_time

print(f"Datenvorbereitung: {data_loading_time:.2f} s")
print(f"Modellaufbau:      {model_creation_time:.2f} s")
print(f"Training:          {training_time:.2f} s")
print(f"Evaluation:        {evaluation_time:.2f} s")
print("-----------------------------")
print(f"Gesamt:            {total_time:.2f} s")

Datenvorbereitung: 0.30 s
Modellaufbau:      2.58 s
Training:          29.81 s
Evaluation:        4.36 s
-----------------------------
Gesamt:            37.04 s
